In [ ]:
from destruction_utilities import *
import numpy as np
import zarr
from tensorflow.keras.models import load_model
from tensorflow.keras.metrics import CategoricalAccuracy, Precision, AUC
import shutil
from tensorflow.keras.utils import Sequence
import os


In [ ]:
CITY = 'test'
TILE_SIZE = (128,128)
THRESHOLD = 0.5

model_path = f'../models/{CITY}/cnn'
images = search_data(pattern(city=CITY, type='image'))
best_model = load_model(model_path, custom_objects={'f1_m':f1_m, 'precision_m': precision_m, 'recall_m': recall_m, 'auc': auc})

In [ ]:
class CNNTestGenerator(Sequence):
    def __init__(self, images, batch_size=32):
        self.images = images
        self.batch_size = batch_size
        
    def __len__(self):
        return len(self.images)//self.batch_size
    
    def __getitem__(self, index):
        if(index == self.__len__()-1):
            X = self.images[index*self.batch_size:len(self.images)]
        else:
            X = self.images[index*self.batch_size:(index+1)*self.batch_size]            
        return X
    
    def augment(self, X):        
        # Brightness
        alpha = random.choice(np.linspace(0.85, 1.4))
        X = X * alpha
        
        return X
    
for image in images:
    profile = tiled_profile(image, tile_size=(*TILE_SIZE, 1))
    name = image.split("image_")[1]
    image = np.array(read_raster(images[3]))
    h,w,b = image.shape
    image = image.reshape(1,h,w,b)
    image = tile_sequences(image, tile_size=TILE_SIZE)   
    image = image.reshape(image.shape[0], image.shape[2], image.shape[3], image.shape[4])
    if os.path.exists('temp.zarr'):
        shutil.rmtree("temp.zarr")
    zarr.save("temp.zarr", np.empty((0,image.shape[1], image.shape[2], image.shape[3])))
    temp = zarr.open("temp.zarr", mode = 'a')
    temp.append(image)
    
    test_images = CNNTestGenerator(temp)
    labels = best_model.predict_generator(test_images)
    labels = np.greater(labels, THRESHOLD).astype('int')
    
    
    write_raster(labels.reshape((profile['height'], profile['width'])), profile, f"../data/{CITY}/predictions/prediction_{name}")    
    print(name.split(".tif")[0])
    


In [ ]:
import gc
gc.collect(generation=2)

In [ ]:
image=images[3]

In [ ]:
labels

In [ ]:
h

In [ ]:
w